In [ ]:
import os
import shutil
import warnings

import matplotlib.pyplot as plt
import scipy.sparse as sp
import scanpy as sc
import squidpy as sq
import harmonypy as hm
import lamindb as ln

warnings.filterwarnings('ignore')

ln.track("nalZv3ddQrdT")

In [ ]:
FIGURES_DIR = '/home/workspace/PCD_data/xenium/fh1_pretreatment/figures/processing'

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
def save_fig(fig, stem, description, ulabels=(), features=None, subdir=''):
    output_dir = os.path.join(FIGURES_DIR, subdir)
    os.makedirs(output_dir, exist_ok=True)
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    for ext in ['png', 'pdf']:
        path = os.path.join(output_dir, f'{stem}.{ext}')
        key = os.path.join(subdir, f'{stem}.{ext}')
        fig.savefig(path, dpi=150, bbox_inches='tight')
        artifact = ln.Artifact(path, key=key, description=description).save()
        if ulabels:
            artifact.ulabels.add(*ulabels)
        if features:
            artifact.features.add_values(features)
    plt.close(fig)

In [ ]:
# ln.Artifact.filter(key__contains='h5ad').df()
# ln.Artifact.filter(key="test_combined_filtered.h5ad").df()

In [ ]:
combined = ln.Artifact.get("9FrxD0fbe2MT8PMj0001").load()

In [ ]:
processing_label  = ln.ULabel(name='processing').save()
umap_label        = ln.ULabel(name='umap').save()
clustered_label   = ln.ULabel(name='clustered').save()

In [ ]:
samples = combined.obs['sample'].unique()
ncols = 4
nrows = (3 + len(samples) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.flatten()

for i, metric in enumerate(['total_counts', 'n_genes_by_counts']):
    sc.pl.umap(combined, color=metric, ax=axes[i], size=6, show=False)

sc.pl.umap(combined, color='sample', ax=axes[2], size=6,
           show=False, legend_loc='lower left')

for i, s in enumerate(samples):
    sc.pl.umap(combined, color='sample', groups=[s], ax=axes[i+3],
               size=6, show=False, title=s, na_color='#d3d3d3', legend_loc=None)
    
for ax in axes[3+len(samples):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

save_fig(fig,
         stem='fh1_pretreatment_umap_qc_sample_overview',
         description='FH1 pretreatment UMAP overview of QC metrics and sample composition, with per-sample highlights.',
         ulabels=(processing_label, umap_label),
         features={'harmony': True, 'spread': 2.0})

In [ ]:
# Register Param schemas once (not inside the loop); these are referenced by features.add_values below.
ln.Param(name='resolution',     dtype=float).save()
ln.Param(name='deg_method',     dtype=str).save()
ln.Param(name='clustering_key', dtype=str).save()
ln.Param(name='n_clusters',     dtype=int).save()
ln.Param(name='harmony',        dtype=bool).save()
ln.Param(name='n_neighbors',    dtype=int).save()
ln.Param(name='n_pcs',          dtype=int).save()
ln.Param(name='spread',         dtype=float).save()

In [ ]:
RESOLUTIONS = [round(0.1 * i, 2) for i in range(1, 12)]
sweep_keys = [f'leiden_res{res:.2f}' for res in RESOLUTIONS]

fig = sc.pl.umap(combined, color=sweep_keys, wspace=0.05, ncols=3,
                 legend_loc='on data', legend_fontoutline=2, show=False, return_fig=True, size=6)

plt.show()

save_fig(fig,
         stem='fh1_pretreatment_umap_resolution_sweep',
         description='FH1 pretreatment UMAP faceted by all leiden resolutions; used to select final clustering resolution',
         ulabels=(processing_label, umap_label),
         features={'harmony': True, 'spread': 2.0})

In [ ]:
highlight_ncols = 4

for res, key in zip(RESOLUTIONS, sweep_keys):
    clusters = combined.obs[key].cat.categories.tolist()
    n_panels = len(clusters) + 1
    highlight_nrows = (n_panels + highlight_ncols - 1) // highlight_ncols
    fig, axes = plt.subplots(highlight_nrows, highlight_ncols,
                             figsize=(5 * highlight_ncols, 5 * highlight_nrows))
    axes = axes.flatten()

    # Plot only: all clusters labeled on the data. Adds uns['{key}_colors'] if missing (already there from the sweep UMAP).
    sc.pl.umap(combined, color=key, ax=axes[0], size=6, show=False,
               legend_loc='on data', title=f'res {res:.2f}: all clusters')

    for i, cluster in enumerate(clusters):
        # Plot only: one cluster in its color, all other cells gray; reuses uns['{key}_colors'].
        sc.pl.umap(combined, color=key, groups=[cluster], ax=axes[i + 1], size=6, show=False,
                   title=f'res {res:.2f}: cluster {cluster}', na_color='#d3d3d3', legend_loc=None)

    for ax in axes[n_panels:]:
        ax.set_visible(False)

    fig.tight_layout()
    plt.show()

    # Writes cluster_highlights/fh1_pretreatment_umap_cluster_highlights_res{res}.png/.pdf under FIGURES_DIR and registers both as lamindb artifacts.
    save_fig(fig,
             stem=f'fh1_pretreatment_umap_cluster_highlights_res{res:.2f}',
             description=f'FH1 pretreatment UMAP of leiden res {res:.2f} clusters: all labeled, plus each cluster highlighted individually',
             ulabels=(processing_label, umap_label),
             features={'harmony': True, 'spread': 2.0, 'resolution': res, 'clustering_key': key,
                       'n_clusters': len(clusters)},
             subdir='cluster_highlights')

In [ ]:
# SPATIAL_DIR = os.path.join(FIGURES_DIR, 'spatial_clusters')
# os.makedirs(SPATIAL_DIR, exist_ok=True)

# for res, key in zip(RESOLUTIONS, sweep_keys):
#     sq.pl.spatial_scatter(combined, color=key, library_key='sample', shape=None, ncols=4, wspace=0.1, size=2)

    # # Writes spatial_clusters/fh1_pretreatment_spatial_clusters_res{res}.png under FIGURES_DIR (on disk, PNG only, 100 dpi); not registered in lamindb.
    # plt.gcf().savefig(os.path.join(SPATIAL_DIR, f'fh1_pretreatment_spatial_clusters_res{res:.2f}.png'), dpi=100, bbox_inches='tight')
    # plt.show()

In [ ]:
FINAL_KEY = 'leiden_res1.10'

fig = sc.pl.umap(combined, color='sample', legend_loc='right margin',
                 show=False, return_fig=True, size=6, title='', frameon=False)
plt.show()

save_fig(fig,
         stem='fh1_pretreatment_umap_samples',
         description='FH1 pretreatment UMAP colored by sample',
         ulabels=(processing_label, umap_label),
         features={'harmony': True, 'spread': 2.0})

# Make UMAP colored by resolution 1.1 leiden clusters (using lgened_fontoutline=2)
fig = sc.pl.umap(combined, color=FINAL_KEY, legend_loc='on data', legend_fontoutline=2,
                 show=False, return_fig=True, size=6, title='', frameon=False)

ax = fig.axes[0]
clusters = combined.obs[FINAL_KEY].cat.categories
cluster_colors = combined.uns[f'{FINAL_KEY}_colors']
handles = [ax.scatter([], [], color=color, s=40, label=cluster) for cluster, color in zip(clusters, cluster_colors)]
ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1, 0.5), frameon=False, ncol=2)
plt.show()

save_fig(fig,
         stem='fh1_pretreatment_umap_leiden_res1.10',
         description='FH1 pretreatment UMAP colored by leiden clusters at resolution 1.1',
         ulabels=(processing_label, umap_label),
         features={'harmony': True, 'spread': 2.0, 'resolution': 1.1})

In [ ]:
# Writes figures/fh1_pretreatment_figures.zip (on disk), rebuilt from everything under FIGURES_DIR; the figure files and lamindb are not touched.
shutil.make_archive(os.path.dirname(FIGURES_DIR) + '/fh1_pretreatment_figures', 'zip',
                    root_dir=os.path.dirname(FIGURES_DIR), base_dir=os.path.basename(FIGURES_DIR))

In [ ]:
# Writes a new lamindb artifact (key 'fh1_pretreatment_combined_clustered.h5ad'); the QC-filtered input artifact is not touched.
artifact = ln.Artifact.from_anndata(combined, key='fh1_pretreatment_combined_clustered.h5ad',
                                    description='Combined FH1 pretreatment Xenium object after batch correction, clustering, and resolution sweep').save()
artifact.ulabels.add(processing_label, clustered_label)
artifact.features.add_values({'harmony': True, 'n_neighbors': 15, 'n_pcs': 30, 'spread': 2.0})

ln.finish()